# 02 - Model Training (FD001)

**Purpose:** Train the required LSTM model with early stopping using unit-level train/validation split.

**Inputs:** processed arrays from `data/processed/`

**Outputs:** `model/best_model.pth`, `model/training_loss_curve.png`, `model/predicted_vs_actual_rul.png`

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split

from model.lstm_model import LSTMRULModel

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "processed").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_DIR = PROJECT_ROOT / "model"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cpu")
torch.manual_seed(42)
np.random.seed(42)

In [ ]:
X_all = np.load(PROCESSED_DIR / "X_train_sequences.npy")
y_all = np.load(PROCESSED_DIR / "y_train_sequences.npy")
seq_unit_ids = np.load(PROCESSED_DIR / "train_sequence_unit_ids.npy")

unique_units = np.unique(seq_unit_ids)
train_units, val_units = train_test_split(unique_units, test_size=0.2, random_state=42)

# Unit-level split avoids leakage from the same engine appearing in both sets.
train_mask = np.isin(seq_unit_ids, train_units)
val_mask = np.isin(seq_unit_ids, val_units)

X_train, y_train = X_all[train_mask], y_all[train_mask]
X_val, y_val = X_all[val_mask], y_all[val_mask]

print("X_train:", X_train.shape, "| y_train:", y_train.shape)
print("X_val:", X_val.shape, "| y_val:", y_val.shape)

In [ ]:
train_loader = DataLoader(
    TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.float32)),
    batch_size=128,
    shuffle=True
)

val_loader = DataLoader(
    TensorDataset(torch.tensor(X_val, dtype=torch.float32), torch.tensor(y_val, dtype=torch.float32)),
    batch_size=128,
    shuffle=False
)

model = LSTMRULModel(input_size=14, hidden_size=64, num_layers=2, dropout=0.3).to(DEVICE)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

EPOCHS = 50
PATIENCE = 10

train_rmse_history = []
val_rmse_history = []
best_val_rmse = float("inf")
no_improve_epochs = 0

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_losses = []

    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        preds = model(xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())

    train_rmse = float(np.sqrt(np.mean(train_losses)))

    model.eval()
    val_losses = []
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            preds = model(xb)
            val_losses.append(criterion(preds, yb).item())

    val_rmse = float(np.sqrt(np.mean(val_losses)))

    train_rmse_history.append(train_rmse)
    val_rmse_history.append(val_rmse)

    if val_rmse < best_val_rmse:
        best_val_rmse = val_rmse
        no_improve_epochs = 0
        torch.save(model.state_dict(), MODEL_DIR / "best_model.pth")
    else:
        no_improve_epochs += 1

    if epoch == 1 or epoch % 5 == 0:
        print(f"Epoch {epoch:02d} | Train RMSE: {train_rmse:.4f} | Val RMSE: {val_rmse:.4f}")

    if no_improve_epochs >= PATIENCE:
        print(f"Early stopping triggered at epoch {epoch}.")
        break

print("Best validation RMSE:", round(best_val_rmse, 4))

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(train_rmse_history, label="Train RMSE", color="#2563eb")
plt.plot(val_rmse_history, label="Validation RMSE", color="#f59e0b")
plt.title("Training vs Validation Loss (RMSE)")
plt.xlabel("Epoch")
plt.ylabel("RMSE")
plt.legend()
plt.tight_layout()
plt.savefig(MODEL_DIR / "training_loss_curve.png", dpi=150)
plt.show()

In [ ]:
best_model = LSTMRULModel(input_size=14, hidden_size=64, num_layers=2, dropout=0.3).to(DEVICE)
best_model.load_state_dict(torch.load(MODEL_DIR / "best_model.pth", map_location=DEVICE))
best_model.eval()

with torch.no_grad():
    val_preds = best_model(torch.tensor(X_val, dtype=torch.float32, device=DEVICE)).cpu().numpy()

plt.figure(figsize=(6, 6))
plt.scatter(y_val, val_preds, alpha=0.5, s=15, color="#f59e0b")
lims = [0, max(float(y_val.max()), float(val_preds.max()))]
plt.plot(lims, lims, color="red", linestyle="--", linewidth=1.5)
plt.title("Predicted vs Actual RUL (Validation)")
plt.xlabel("Actual RUL")
plt.ylabel("Predicted RUL")
plt.tight_layout()
plt.savefig(MODEL_DIR / "predicted_vs_actual_rul.png", dpi=150)
plt.show()

print("Saved checkpoint:", MODEL_DIR / "best_model.pth")